In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler, Dataset, DataLoader
import torchvision.transforms as T

# Train df

In [ ]:
com_train_df = pd.read_csv('/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv')
com_train_df

# Test df

In [ ]:
test_df = pd.read_csv('/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/test.csv')
test_df

# Sample_submission df

In [ ]:
sample_df = pd.read_csv('/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/sample_submission.csv')
sample_df

# Count of each class

In [ ]:
label_counts = com_train_df.iloc[:, 1:].sum()
label_counts

In [ ]:
plt.plot(label_counts, 'o-')
plt.xticks(rotation=75)
plt.show()

# Column names of com_train_df

In [ ]:
com_train_df.columns

# Image file directories

In [ ]:
image_dir = Path('/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/images')
image_path_list = list(image_dir.glob('*.png'))
print(f'Total images: {len(image_path_list)}')

# Image insights

In [ ]:
img = Image.open(image_path_list[0])
print(f'Image mode is: {img.mode}')
img = np.array(img)
print(f'Image shape: {img.shape}')
print(f'pixel values ranges from {img.min()}-{img.max()}')
print('-----min, max pixel value in each channel of image[0]-----')
print(f'{'':<3}{'min':<5}{'max':<5}')
print('='*20)
print(f'{'R':<3}{img[:, :, 0].min():<5}{img[:, :, 0].max():<5}')
print(f'{'G':<3}{img[:, :, 1].min():<5}{img[:, :, 1].max():<5}')
print(f'{'B':<3}{img[:, :, 2].min():<5}{img[:, :, 2].max():<5}')

# Visualizing Images

In [ ]:
plt.figure(figsize=(15, 12))
for i in range(3):
    plt.subplot(1, 3, i+1)
    img = Image.open(image_path_list[i])
    plt.imshow(img)
    plt.axis(False)
plt.show()

# Setting up the device

In [ ]:
device = 'cpu'
print(f'CUDA is available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')
    print(f'Device properties: {torch.cuda.get_device_properties(0)}')
    device = 'cuda'

# Custom Dataset

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, img_files: np.array, labels: np.array, img_dir: Path, transform=None):
        self.img_files = img_files
        self.labels = labels
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return self.img_files.shape[0]
    def __getitem__(self, index):
        img = Image.open(self.img_dir / self.img_files[index])
        label = self.labels[index]

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
img_files = np.array(com_train_df.iloc[:, 0].values)
label_names = com_train_df.columns[1:]
labels = np.array([np.argmax(com_train_df.iloc[i, 1:]) for i in range(com_train_df.shape[0])])
img_files.shape, labels.shape

# Train val split

In [ ]:
train_img_files, val_img_files, train_labels, val_labels = train_test_split(img_files, labels, test_size=0.2, random_state=42)
len(train_img_files), len(val_img_files), len(train_labels), len(val_labels)

# Mean and Std of each channel of training images

In [ ]:
def get_mean_std(train_img_files, labels, img_dir):
    temp_transform = T.Compose([
        T.Resize((224, 224)), 
        T.ToTensor()
    ])
    train_dataset = ImageDataset(train_img_files, labels, img_dir, transform=temp_transform)
    dataloader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=2)
    sum_ = 0.
    sum_sq = 0.
    num_pixels = 0
    for X, _ in dataloader:
        X = X.view(X.size(0), X.size(1), -1)
        sum_ += X.sum(dim=[0, 2])
        sum_sq += (X ** 2).sum(dim=[0, 2])

        num_pixels += X.size(0) * X.size(2)

    mean = sum_/num_pixels
    std = torch.sqrt(sum_sq/num_pixels - mean**2)

    return mean, std

In [ ]:
DATASET_MEAN, DATASET_STD = get_mean_std(train_img_files, labels, image_dir)
DATASET_MEAN, DATASET_STD

In [ ]:
# count of each class
label_c_tensor = torch.tensor(label_counts.values)

# label_weights = 1.0 / label_c_tensor
# sampler = WeightedRandomSampler(weights=label_weights[train_labels], num_samples=len(train_labels))

# Cost Matrix

In [ ]:
num_classes = 20
cost_matrix = torch.zeros((20, 20)).to(device)
for true in range(num_classes):
    for pred in range(num_classes):
        if true == pred:
            cost_matrix[true, pred] = -1
        else:
            if pred == 19:
                cost_matrix[true, pred] = 5
            elif true == 19:
                cost_matrix[true, pred] = 1
            else:
                cost_matrix[true, pred] = 6

In [ ]:
import time
from tqdm.notebook import tqdm

# Pretrained model

Let's try using pretrained models.\
We will use DenseNet121

## DenseNet121

In [ ]:
import torchvision.models as models

In [ ]:
WEIGHTS = models.DenseNet121_Weights.DEFAULT
model = models.densenet121(weights=WEIGHTS)
model

In [ ]:
from torchinfo import summary

summary(model=model, input_size=(64, 3, 224, 224))

# Setting up the model parameters, optimizer and the loss function

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.features.denseblock3.parameters():
    param.requires_grad = True
    
for param in model.features.denseblock4.parameters():
    param.requires_grad = True

model.classifier = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Dropout(p=0.5), nn.Linear(128, 20))
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
# criterion = nn.CrossEntropyLoss(weight=(label_weights*len(train_img_files)).to(device))
criterion = nn.CrossEntropyLoss()

In [ ]:
# checking the expected transformations
auto_transform = WEIGHTS.transforms()
auto_transform

# Transformations

In [ ]:
train_transform = T.Compose([
    T.Resize((224, 224)), 
    T.RandomHorizontalFlip(p=0.5), 
    T.RandomRotation(degrees=10), 
    T.ToTensor(), 
    T.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
    # T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = T.Compose([
    T.Resize((224, 224)), 
    T.ToTensor(), 
    T.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
    # T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])

# Setting up dataloaders

In [ ]:
train_dataset = ImageDataset(train_img_files, train_labels, image_dir, transform=train_transform)
val_dataset = ImageDataset(val_img_files, val_labels, image_dir, transform=val_transform)

# train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=2)
train_loader = DataLoader(train_dataset, batch_size=64, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# X, y = next(iter(train_loader))

# for i in range(200):
#     outputs = model(X.to(device))
#     loss = criterion(outputs, y.to(device))

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     if i % 50 == 0:
#         print(loss.item())

# Training Function

In [ ]:
def train_loop(train_loader, model, criterion, optimizer, cost_matrix, device, num_epochs=10):
    model.train()
    
    train_losses = []
    train_losses_mat = []
    train_accuracies = []

    
    start_time = time.time()

    
    for epoch in range(num_epochs):
        
    
        running_loss = 0.
        running_loss_mat = 0.
        correct = 0
        total = 0
    
        progress_bar = tqdm(enumerate(train_loader), desc=f'Epoch {epoch+1}/{num_epochs}', total=len(train_loader))
        
        for i, (X, y) in progress_bar:
            X, y = X.to(device), y.to(device)
            
            outputs = model(X)
            loss = criterion(outputs, y)
            
            optimizer.zero_grad()
            
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += y.eq(predicted).sum().item()
            for j in range(y.size(0)):
                running_loss_mat += cost_matrix[y[j], predicted[j]].item()
            total += y.size(0)
            
            # Update progress bar
            avg_loss = running_loss/(i+1)
            accuracy = 100. * correct/total
            progress_bar.set_postfix({
                'loss': f'{avg_loss:.4f}',
                'loss_matrix': f'{running_loss_mat / total :.4f}', 
                'accuracy': f'{accuracy:.2f}%'
            })
            
            epoch_loss = running_loss / len(train_loader)
            epoch_accuracy = 100. * correct / total
            
            train_losses.append(epoch_loss)
            train_losses_mat.append(running_loss_mat/total)
            train_accuracies.append(epoch_accuracy)

    
            
    print(f'Training completed in {time.time()-start_time:.2f} seconds')
    return train_losses, train_losses_mat, train_accuracies

# Validation Function

In [ ]:
def val_loop(val_loader, model, criterion, cost_matrix, device):
    model.eval()
    
    val_loss = 0.
    val_loss_mat = 0.
    val_accuracy = 0.
   
    running_loss = 0.
    running_loss_mat = 0.
    correct = 0
    total = 0
        
    for i, (X, y) in enumerate(val_loader):
        X, y = X.to(device), y.to(device)
        
        outputs = model(X)
        loss = criterion(outputs, y)
        

        running_loss += loss.item() * y.size(0)
        _, predicted = outputs.max(1)
        correct += y.eq(predicted).sum().item()
        for j in range(y.size(0)):
            running_loss_mat += cost_matrix[y[j], predicted[j]].item()
        total += y.size(0)
        
            
    val_loss = running_loss / total
    val_loss_mat = running_loss_mat / total
    val_accuracy = 100. * correct / total
            
    return val_loss, val_loss_mat, val_accuracy

# Training

In [ ]:
train_losses, train_losses_mat, train_accuracies = train_loop(train_loader, model, criterion, optimizer, cost_matrix, device, 5)

# Validation

In [ ]:
val_loss, val_loss_mat, val_accuracy = val_loop(val_loader, model, criterion, cost_matrix, device)

print(f'validation loss(CrossEntropy): {val_loss:.4f}')
print(f'validation loss(cost_matrix): {val_loss_mat:.4f}')
print(f'validation accuracy: {val_accuracy:.4f}')

For 1 epoch In DenseNet

train_loss, train_loss_mat, train_accuracy \
case 1: CrossEntropyLoss no weights, sampler\
2.1614--      3.2423--          35.80 \
case 2: CrossEntropyLoss weights, sampler\
0.5328--      4.0080--          24.89 \
case 3: CrossEntropyLoss no weights, no sampler\
1.3520--      1.0137--          65.63 \
case 4: CrossEntropyLoss weights, no sampler\
2.7823--      2.1569--          14.13

val_loss, val_loss_mat, val_accuracy \
case 1: CrossEntropyLoss no weights, sampler\
2.499--      2.032--         16.68 \
case 2: CrossEntropyLoss weights, sampler\
3.547--      2.621--         0.6563 \
case 3: CrossEntropyLoss no weights, no sampler\
1.253--      0.978--         66.46 \
case 4: CrossEntropyLoss weights, no sampler\
2.659--      1.880--         23.49

# Submission

In [ ]:
test_file_names = test_df.iloc[:, 0].values
test_labels = []
model.eval()
for file in test_file_names:
    img = Image.open(image_dir / file)
    transformed_img = val_transform(img)
    outputs = model(transformed_img.unsqueeze(0).to(device))
    _, preds = outputs.max(1)
    test_labels.append(preds.squeeze().item())

ohe_array = np.zeros((len(test_file_names), 20))
for i, label in enumerate(test_labels):
    ohe_array[i, label] = 1


submission_df = pd.DataFrame(ohe_array, columns=com_train_df.columns[1:])
submission_df.insert(0, com_train_df.columns[0], test_file_names)

submission_df.to_csv('submission.csv', index=False)